Cell 1 — Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting"
)

ADVANCED_DIR = BASE_DIR / "data" / "advanced"

Cell 2 — Load current best dataset

In [2]:
train = pd.read_csv(
    ADVANCED_DIR / "train_expanded_ewma_seasonal.csv"
)

train["date"] = pd.to_datetime(train["date"])

print(train.shape)

(3000888, 57)


Cell 3 — Use log-sales for aggregate statistics

In [3]:
train["log_sales"] = np.log1p(train["sales"])

Cell 4 — Family aggregates

In [4]:
family_stats = (
    train.groupby("family")["log_sales"]
    .agg(["mean", "std"])
    .reset_index()
)

family_stats.columns = [
    "family",
    "family_target_mean",
    "family_target_std"
]

train = train.merge(
    family_stats,
    on="family",
    how="left"
)

print(train.shape)

(3000888, 60)


Cell 5 — Store aggregates

In [5]:
store_stats = (
    train.groupby("store_nbr")["log_sales"]
    .agg(["mean", "std"])
    .reset_index()
)

store_stats.columns = [
    "store_nbr",
    "store_target_mean",
    "store_target_std"
]

train = train.merge(
    store_stats,
    on="store_nbr",
    how="left"
)

print(train.shape)

(3000888, 62)


Cell 6 — Store-family aggregates

In [6]:
store_family_stats = (
    train.groupby(["store_nbr", "family"])["log_sales"]
    .agg(["mean", "std"])
    .reset_index()
)

store_family_stats.columns = [
    "store_nbr",
    "family",
    "store_family_target_mean",
    "store_family_target_std"
]

train = train.merge(
    store_family_stats,
    on=["store_nbr", "family"],
    how="left"
)

print(train.shape)

(3000888, 64)


Cell 7 — City / state / cluster aggregates

In [7]:
for col in ["city", "state", "cluster", "type"]:
    
    stats = (
        train.groupby(col)["log_sales"]
        .agg(["mean", "std"])
        .reset_index()
    )
    
    stats.columns = [
        col,
        f"{col}_target_mean",
        f"{col}_target_std"
    ]
    
    train = train.merge(
        stats,
        on=col,
        how="left"
    )

print(train.shape)

(3000888, 72)


Cell 8 — Fill missing aggregate std values

In [8]:
agg_features = [
    col for col in train.columns
    if "target_mean" in col
    or "target_std" in col
]

train[agg_features] = train[agg_features].fillna(0)

print(train[agg_features].isnull().sum().sum())
print(len(agg_features))

0
14


Cell 9 — Drop helper column and save

In [9]:
train = train.drop(columns=["log_sales"])

train.to_csv(
    ADVANCED_DIR / "train_expanded_ewma_seasonal_aggregates.csv",
    index=False
)

print(train.shape)

(3000888, 71)
